In [13]:
from matipo import SEQUENCE_DIR, GLOBALS_DIR
from matipo.sequence import Sequence
import numpy as np
import panel as pn
import yaml
import logging
from pathlib import Path
import pandas as pd

# Compile parameters into yaml
Compiles the df_fitE and df_fitB0 into a yaml that is saved into globals: eddy_E_fit.yaml and eddy_B0_fit.yaml

In [31]:
def empty_fit():
    return {
        "A0": 0,
        "A1": 0,
        "tau1": 0,
        "A2": 0,
        "tau2": 0,
        "A3": 0,
        "tau3": 0,
        "A4": 0,
        "tau4": 0,
    }

def compile_fitting_pars(measure_folder, globals_path):
    """
    """
    # get path to the data folder from notebook
    p = Path.cwd()
    
    while p != p.parent:
        if (p / "data").exists():
            print("Found:", p / "data")
            break
        p = p.parent
    
    data_path = p
    

    
    gradient_summary = {}
    offset_summary = {}
    empty_axes = []
    for axis in ["X", "Y", "Z"]:
    
        folder = data_path / 'data'/ measure_folder / f"{axis}grad_{axis}eddy"
        print(folder)
    
        if not folder.exists(): # if it hasn't been measured then will just be zeros instead
            print(folder, 'doesnt exist')
            gradient_summary[axis] = empty_fit()
            offset_summary[axis] = empty_fit()
            empty_axes.append(axis)
            continue
    
        # -------- E fit --------
        fit_E_file = folder / "df_fitE.csv"
    
        if fit_E_file.exists():
            df_E = pd.read_csv(fit_E_file)
            print(fit_E_file)
            print(df_E)
            gradient_summary[axis] = {
                row["Parameter"]: float(row["df_fit"])
                for _, row in df_E.iterrows()
    
            }
        else:
            gradient_summary[axis] = empty_fit()
    
        # -------- B0 fit --------
        fit_B0_file = folder / "df_fitB0.csv"
    
        if fit_B0_file.exists():
            df_B0 = pd.read_csv(fit_B0_file)
            print(df_B0)
    
            offset_summary[axis] = {
                row["Parameter"]: float(row["df_fit"])
                for _, row in df_B0.iterrows()
    
            }
        else:
            empty_axes.append(axis)
            offset_summary[axis] = empty_fit()
    
    globals_path = Path(globals_path)
    
    with open(globals_path / "eddy_E_fit.yaml", "w") as f:
        yaml.safe_dump(gradient_summary, f, sort_keys=False, default_flow_style=False)
    
    with open(globals_path / "eddy_B0_fit.yaml", "w") as f:
        yaml.safe_dump(offset_summary, f, sort_keys=False, default_flow_style=False)

    return empty_axes


In [32]:
compile_fitting_pars("Measure_EC", GLOBALS_DIR)

Found: C:\Users\halesda\OneDrive - Victoria University of Wellington - STAFF\Desktop\kahu-workspace\data
C:\Users\halesda\OneDrive - Victoria University of Wellington - STAFF\Desktop\kahu-workspace\data\Measure_EC\Xgrad_Xeddy
C:\Users\halesda\OneDrive - Victoria University of Wellington - STAFF\Desktop\kahu-workspace\data\Measure_EC\Xgrad_Xeddy\df_fitE.csv
  Parameter    df_fit
0        A0 -0.002202
1        A1  0.062856
2      tau1  0.272118
3        A2  0.056673
4      tau2  0.053684
5        A3  0.037123
6      tau3  0.006896
7        A4  0.000000
8      tau4  0.000000
  Parameter    df_fit
0        A0 -0.000018
1        A1 -0.046449
2      tau1  0.047002
3        A2  0.045978
4      tau2  0.045761
5        A3  0.000000
6      tau3  0.000000
7        A4  0.000000
8      tau4  0.000000
C:\Users\halesda\OneDrive - Victoria University of Wellington - STAFF\Desktop\kahu-workspace\data\Measure_EC\Ygrad_Yeddy
C:\Users\halesda\OneDrive - Victoria University of Wellington - STAFF\Desktop\ka

[]